In [1]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus
import os
from dotenv import load_dotenv

load_dotenv(dotenv_path='../.env')

db_user = os.getenv('DB_USER')
db_password = quote_plus(os.getenv('DB_PASSWORD'))  # encodes special characters like @
db_host = os.getenv('DB_HOST')
db_port = os.getenv('DB_PORT')
db_name = os.getenv('DB_NAME')

engine = create_engine(f'postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}')


In [4]:
import pandas as pd

claims_with_bene = pd.read_csv('../data/processed/claims_with_beneficiary.csv')

# Lowercase column names to match standard Postgres convention
claims_with_bene.columns = [c.lower() for c in claims_with_bene.columns]

# Columns that are numeric-coded flags (1.0/2.0 style) — need Int64 cleanup
numeric_flag_cols = ['bene_sex_ident_cd', 'bene_race_cd',
                      'sp_alzhdmta', 'sp_chf', 'sp_chrnkidn', 'sp_cncr', 'sp_copd',
                      'sp_depressn', 'sp_diabetes', 'sp_ischmcht', 'sp_osteoprs',
                      'sp_ra_oa', 'sp_strketia']

for col in numeric_flag_cols:
    claims_with_bene[col] = claims_with_bene[col].astype('Int64').astype(str).replace('<NA>', None)

# bene_esrd_ind is already 'Y'/'0' text — just make sure it's clean string, no numeric cast
claims_with_bene['bene_esrd_ind'] = claims_with_bene['bene_esrd_ind'].astype(str).replace('nan', None)

claims_with_bene.to_sql('claims_with_beneficiary', engine, if_exists='append', index=False, chunksize=5000)

print("Load complete. Rows loaded:", len(claims_with_bene))

Load complete. Rows loaded: 857563
